In [1]:

import pandas as pd
import numpy as np
import statsmodels.api as sm

def calcular_alfa_jensen_seguro(df_resultados):
    df = df_resultados.copy()
    
    # 1. Identifica a coluna do Benchmark dinamicamente
    col_bench = None
    for candidata in ['Retorno_Benchmark', 'Retorno_Ibov', 'Retorno_Ibovespa', 'Benchmark']:
        if candidata in df.columns:
            col_bench = candidata
            break
            
    if col_bench is None:
        raise KeyError(f"Nenhuma coluna de benchmark encontrada. Colunas presentes: {df.columns.tolist()}")

    # 2. Identifica ou trata a coluna do CDI (Risk-Free)
    if 'Retorno_CDI' in df.columns:
        col_cdi = df['Retorno_CDI']
    elif 'CDI' in df.columns:
        col_cdi = df['CDI']
    else:
        col_cdi = pd.Series((1 + 0.10)**(1/252) - 1, index=df.index)

    col_modelo = 'Retorno_Modelo' if 'Retorno_Modelo' in df.columns else df.columns[0]

    excess_mod = (df[col_modelo] - col_cdi).dropna()
    excess_bench = (df[col_bench] - col_cdi).dropna()

    idx_comum = excess_mod.index.intersection(excess_bench.index)
    y = excess_mod.loc[idx_comum]
    x = excess_bench.loc[idx_comum]

    X = sm.add_constant(x)
    reg = sm.OLS(y, X).fit()

    alpha_diario = reg.params.iloc[0]  # const
    beta = reg.params.iloc[1]          # coeficiente angular (Beta)
    p_alpha = reg.pvalues.iloc[0]
    p_beta = reg.pvalues.iloc[1]
    r2 = reg.rsquared

    n_dias = len(y)
    cagr_mod = (np.prod(1 + df[col_modelo].loc[idx_comum]) ** (252 / n_dias)) - 1
    cagr_bench = (np.prod(1 + df[col_bench].loc[idx_comum]) ** (252 / n_dias)) - 1
    cagr_cdi = (np.prod(1 + col_cdi.loc[idx_comum]) ** (252 / n_dias)) - 1
    
    alfa_anual = ((1 + alpha_diario) ** 252) - 1
    retorno_esperado_capm = cagr_cdi + beta * (cagr_bench - cagr_cdi)

    print("=" * 55)
    print("        DECOMPOSIÇÃO DE PERFORMANCE (JENSEN CAPM)")
    print("=" * 55)
    print(f"CAGR Modelo Total:            {cagr_mod:.2%}")
    print(f"CAGR Benchmark ({col_bench}): {cagr_bench:.2%}")
    print(f"CAGR CDI (Risk-Free):         {cagr_cdi:.2%}")
    print("-" * 55)
    print(f"Beta do Portfólio (β):        {beta:.4f} (p-valor: {p_beta:.4f})")
    print(f"R² com o Mercado:             {r2:.2%}")
    print(f"Retorno Esperado pelo Risco:  {retorno_esperado_capm:.2%}")
    print(f"Alfa de Jensen Real (α):      {alfa_anual:.2%} (p-valor: {p_alpha:.4f})")
    print(f"Excesso Bruto Simples:        {(cagr_mod - cagr_bench):.2%}")
    print("=" * 55)

    return {
        "cagr_modelo": cagr_mod,
        "cagr_benchmark": cagr_bench,
        "cagr_cdi": cagr_cdi,
        "beta": beta,
        "r2": r2,
        "alfa_anual": alfa_anual,
        "p_valor_alfa": p_alpha
    }

df_resultado = pd.read_csv('etl/df_resultados.csv')
calcular_alfa_jensen_seguro(df_resultados=df_resultado)

        DECOMPOSIÇÃO DE PERFORMANCE (JENSEN CAPM)
CAGR Modelo Total:            13.23%
CAGR Benchmark (Retorno_Benchmark): 5.59%
CAGR CDI (Risk-Free):         9.59%
-------------------------------------------------------
Beta do Portfólio (β):        -0.0432 (p-valor: 0.0007)
R² com o Mercado:             0.77%
Retorno Esperado pelo Risco:  9.77%
Alfa de Jensen Real (α):      4.04% (p-valor: 0.4234)
Excesso Bruto Simples:        7.63%


{'cagr_modelo': np.float64(0.13226704073046225),
 'cagr_benchmark': np.float64(0.055929229146934656),
 'cagr_cdi': np.float64(0.0959262480486811),
 'beta': np.float64(-0.04324089688681211),
 'r2': np.float64(0.0076541593194986435),
 'alfa_anual': np.float64(0.04039096350064364),
 'p_valor_alfa': np.float64(0.4233802230591003)}

In [2]:
trocas_reais = (df_resultado['Evento'] == 'ORDEM_CRIADA_REBALANCEAMENTO').sum()
print(f"Total de rebalanceamentos em 10 anos: {trocas_reais}")
print(f"Custo total acumulado de transação: R$ {df_resultado['Custo_Transacao'].sum():,.2f}")

Total de rebalanceamentos em 10 anos: 54
Custo total acumulado de transação: R$ 27,764.72


In [3]:
def analisar_resultados_por_regime(df):
    """
    Agrupa e calcula métricas institucionais de retorno, volatilidade, 
    eficiência e exposição física para cada regime predito pelo HMM.
    """
    df = df.sort_index()
    metricas_regimes = []
    regimes_unicos = df['Regime_Macro'].dropna().unique()
    
    for regime in regimes_unicos:
        df_sub = df[df['Regime_Macro'] == regime]
        
        if len(df_sub) < 5:
            continue
        retornos = df_sub['Retorno_Modelo'].dropna()        
        total_dias = len(df_sub)
        porcentagem_tempo = (total_dias / len(df)) * 100
        retorno_medio_anual = (1 + retornos.mean()) ** 252 - 1        
        vol_anual = retornos.std() * np.sqrt(252)
        sharpe = retorno_medio_anual / vol_anual if vol_anual != 0 else 0
        if 'Exposicao' in df_sub.columns:
            exp_media = df_sub['Exposicao_Acoes'].mean()
        elif 'Capital_Acoes' in df_sub.columns and 'Patrimonio' in df_sub.columns:
            exp_media = (df_sub['Capital_Acoes'] / df_sub['Patrimonio']).mean()
        else:
            exp_media = np.nan
            
        metricas_regimes.append({
            "Regime": int(regime),
            "Dias Ativo": total_dias,
            "% Tempo Fundo": f"{porcentagem_tempo:.1f}%",
            "Retorno Médio Anual": f"{retorno_medio_anual * 100:.2f}%",
            "Volatilidade Anual": f"{vol_anual * 100:.2f}%",
            "Sharpe Ratio": f"{sharpe:.2f}",
            "Exposição Média": f"{exp_media * 100:.2f}%" if not np.isnan(exp_media) else "N/A"
        })
        
    df_analise = pd.DataFrame(metricas_regimes).sort_values(by="Regime")
    
    print("\n" + "="*25 + " RAIO-X DE PERFORMANCE POR REGIME (HMM) " + "="*25)
    print(df_analise.to_string(index=False))
    print("="*90)
    
    return df_analise

df_regimes_summary = analisar_resultados_por_regime(df_resultado)


========================= RAIO-X DE PERFORMANCE POR REGIME (HMM) =========================
 Regime  Dias Ativo % Tempo Fundo Retorno Médio Anual Volatilidade Anual Sharpe Ratio Exposição Média
      0         463         31.1%              22.54%              9.68%         2.33          38.05%
      1         491         33.0%              11.72%             13.34%         0.88          45.59%
      2         371         24.9%              11.84%             13.21%         0.90          41.15%
      3         163         11.0%               3.50%             11.25%         0.31          40.90%


In [9]:
import plotly.graph_objects as go
import pandas as pd

def plot_evolucao_exposicao_com_regimes(df_resultado):
    """
    Gera um gráfico dinâmico de área empilhada (Ações vs CDI)
    com faixas coloridas no fundo representando os Regimes do HMM.
    """
    df_plot = df_resultado.copy()
    
    # Garante a escala de 0 a 100% para o gráfico
    df_plot['Exposicao_Acoes_Pct'] = df_plot['Exposicao_Acoes'] * 100
    df_plot['Exposicao_CDI_Pct'] = 100 - df_plot['Exposicao_Acoes_Pct']
    
    fig = go.Figure()
    
    # 1. Camada de Área Empilhada: Caixa (CDI)
    fig.add_trace(go.Scatter(
        x=df_plot.index, 
        y=df_plot['Exposicao_CDI_Pct'],
        mode='lines',
        name='Caixa (CDI)',
        stackgroup='one',
        groupnorm='percent',
        marker_color='rgba(230, 230, 230, 0.5)',
        line=dict(width=0.5)
    ))
    
    # 2. Camada de Área Empilhada: Exposição em Ações
    fig.add_trace(go.Scatter(
        x=df_plot.index, 
        y=df_plot['Exposicao_Acoes_Pct'],
        mode='lines',
        name='Exposição em Ações',
        stackgroup='one',
        marker_color='rgba(44, 160, 44, 0.85)', # Azul institucional contínuo
        line=dict(width=1)
    ))

    # =========================================================================
    # --- MAPEAMENTO DOS REGIMES DO HMM (FAIXAS VERTICAIS DE BACKGROUND) ---
    # =========================================================================
    # Configuração de cores suaves (transparentes) para o fundo não cobrir os dados
    cores_regimes = {
        0: 'rgba(46, 204, 113, 0.08)',   # Verde muito suave (Bull_Baixa_Vol)
        1: 'rgba(52, 152, 219, 0.05)',   # Azul muito suave (Transicao_Normal)
        2: 'rgba(241, 196, 15, 0.08)',   # Amarelo muito suave (Correcao)
        3: 'rgba(231, 76, 60, 0.12)'     # Vermelho visível (Crise_Panico)
    }
    
    # Identifica os pontos exatos onde o regime macro mudou na história
    df_plot['Mudou_Regime'] = df_plot['Regime_Macro'].diff().fillna(0) != 0
    datas_mudanca = df_plot[df_plot['Mudou_Regime']].index.tolist()
    
    # Garante os limites inicial e final da série histórica
    datas_limite = [df_plot.index[0]] + datas_mudanca + [df_plot.index[-1]]
    
    # Desenha as caixas (shapes) verticais no layout do Plotly
    shapes = []
    for idx in range(len(datas_limite) - 1):
        data_ini = datas_limite[idx]
        data_fim = datas_limite[idx + 1]
        
        # Pega o regime vigente naquele intervalo de tempo
        regime_vigente = df_plot.loc[data_ini, 'Regime_Macro']
        
        shapes.append(dict(
            type="rect",
            xref="x",
            yref="paper", # Trava o topo e o fundo do shape na moldura do gráfico
            x0=data_ini,
            y0=0,
            x1=data_fim,
            y1=1,
            fillcolor=cores_regimes.get(regime_vigente, 'rgba(0,0,0,0)'),
            line=dict(width=0), # Sem bordas para não poluir
            layer="below" # Força as faixas coloridas a ficarem ATRÁS das áreas empilhadas
        ))
        
    # Adiciona traços invisíveis na legenda apenas para mapear a cor de cada regime pro usuário
    nomes_regimes = {0: "Bull Market", 1: "Transição", 2: "Correção", 3: "Crise/Pânico"}
    for reg, cor in cores_regimes.items():
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=10, color=cor.replace('0.', '0.9'), symbol='square'), # Cor opaca na legenda
            name=f"Regime: {nomes_regimes[reg]}",
            showlegend=True
        ))

    # Configurações estéticas e eixos
    fig.update_layout(
        title='<b>Evolução da Alocação Dinâmica vs Regimes do HMM</b><br><sup>Rotação tática sob estresse: fundos coloridos indicam o regime de mercado determinado pelo modelo</sup>',
        xaxis_title='Tempo',
        yaxis_title='Alocação do Capital (%)',
        hovermode='x unified',
        template='plotly_white',
        shapes=shapes, # Injeta os backgrounds calculados
        yaxis=dict(ticksuffix='%', range=[0, 100]),
        legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5) # Legenda na parte inferior interna
    )
    
    fig.show()

# Chamada da nova função
plot_evolucao_exposicao_com_regimes(df_resultado)